# PyTorch 입문 — 커스텀 Module · 수동 forward

손으로 `y = wx + b`를 쓰고, `nn.Linear`와 `LinearReg`로 같은 식을 감싼 뒤, `model(x)`가 `forward`를 부르는 흐름을 봅니다.  
이론 복습: [`../session08.md`](../session08.md)

> **이 노트북이 하는 일**  
> 수동 forward → `nn.Linear` · `state_dict` → 커스텀 `LinearReg` → `__call__` → 다출력 행렬 · `mse_loss`.

환경: `torch-env` 커널을 쓰는 것을 권장합니다.

## 1. import

In [48]:
import torch
import numpy as np # numpy: 수치 계산 모듈
from torch import nn # torch.nn: 신경망 모듈
import torch.nn.functional as F # torch.nn.functional: 신경망 함수

## 2. 손으로 쓰는 forward

스칼라 `w`, `b`로 `yhat = w * x + b`를 계산합니다.

In [49]:
w = torch.tensor(3.0)  # 0차원 스칼라
b = torch.tensor(-1.0)
print(w, b)

tensor(3.) tensor(-1.)


In [50]:
def forward(x):
    # yhat = wx + b
    yhat = w * x + b
    return yhat

In [51]:
x = torch.tensor(1.0)
y_forward = forward(x)  # 3*1 - 1 = 2
print(y_forward) # tensor(2.)
print(y_forward.item()) # 2.0

tensor(2.)
2.0


## 3. 여러 입력에 같은 forward

각 칸에 같은 식을 적용하면 출력 모양이 입력과 같습니다.

In [52]:
x = torch.tensor([[1.0, 3.0, 5.0]])  # 1 x 3 행렬 [[1.0, 3.0, 5.0]]
yhat = forward(x)
# [[
#  3.0 * 1. + (-1.0) = 2.,
#  3.0 * 3. + (-1.0) = 8.,
#  3.0 * 5. + (-1.0) = 14.
# ]]
print(x.size()) # torch.Size([1, 3])
print(yhat.shape) # torch.Size([1, 3])

torch.Size([1, 3])
torch.Size([1, 3])


## 4. `nn.Linear` · `state_dict`

입력 1 · 출력 1 직선 층입니다. 생성 직후 가중치는 랜덤입니다.

In [53]:
from torch.nn import Linear # torch.nn.Linear: 선형 층

torch.manual_seed(1)

# Linear 함수: forward(x) 와 동일한 식을 계산하는 함수
# in_features: 입력 특성 수, out_features: 출력 특성 수, bias: 편향 포함 여부
model = Linear(in_features=1, out_features=1, bias=True)
# state_dict(): 모델의 가중치와 편향을 저장하는 딕셔너리
print(model.state_dict()) # OrderedDict [('weight', tensor([[0.5153]]), ('bias', tensor([-0.4414])))]
print(model.state_dict()["weight"]) # tensor([[0.5153]])
print(model.state_dict()["bias"]) # tensor([-0.4414])
print(model.weight) # tensor([[0.5153]])
print(model.bias) # tensor([-0.4414])
# requires_grad=True : 이 값이 거치는 계산을 기록해 기울기를 구할 수 있음

OrderedDict([('weight', tensor([[0.5153]])), ('bias', tensor([-0.4414]))])
tensor([[0.5153]])
tensor([-0.4414])
Parameter containing:
tensor([[0.5153]], requires_grad=True)
Parameter containing:
tensor([-0.4414], requires_grad=True)


## 5. 커스텀 Module — `LinearReg`

`nn.Module`을 상속해 `nn.Linear`를 안에 두고, `forward`에서 호출합니다.

In [54]:
class LinearReg(nn.Module):
    def __init__(self, in_size, out_size):
        # 부모 초기화로 가중치 등록·추적을 켠다
        super().__init__()
        self.linear = nn.Linear(in_size, out_size)
        # nn.Linear(in_size, out_size): 입력 특성 수, 출력 특성 수
    def forward(self, x):
        out = self.linear(x)  # out = wx + b
        return out

In [55]:
model = LinearReg(1, 1)
# model.state_dict(): 모델의 가중치와 편향을 저장하는 딕셔너리
model.state_dict()
# OrderedDict([
# ('linear.weight', tensor([[-0.1939]])),
# ('linear.bias', tensor([0.4694]))
# ])
# torch.manual_seed 선언으로 고정된 랜덤 시드 출럭

OrderedDict([('linear.weight', tensor([[-0.1939]])),
             ('linear.bias', tensor([0.4694]))])

## 6. `model(x)` → `__call__` → `forward`

`model(x)`는 `__call__`을 거쳐 `forward`를 실행합니다.

In [56]:
x = torch.tensor([[1.1], [2.1], [3.1]])
yhat = model(x)  # __call__ → forward
#  weight(-0.1939)*x + bias(0.4694)
# -0.1939 * [1.1] + 0.4694 = -0.2133
# -0.1939 * [2.1] + 0.4694 = 0.1712
# -0.1939 * [3.1] + 0.4694 = 0.5557
print(yhat)
# tensor([[-0.2133], [0.1712], [0.5557]], grad_fn=<AddmmBackward>)

tensor([[ 0.2561],
        [ 0.0623],
        [-0.1316]], grad_fn=<AddmmBackward0>)


In [57]:
class A:
    def __call__(self, x):
        return x + 1


a = A()
print(a(3)) # 4

4


## 7. 짧은 기울기 확인

`requires_grad=True`로 추적을 켠 뒤 `backward`로 `.grad`를 채웁니다.

In [58]:
x = torch.rand(1, requires_grad=True) # 1 x 1 행렬 [[random_x]]
y = torch.rand(1, requires_grad=True) # 1 x 1 행렬 [[random_y]]
loss = y - x # random_y - random_x
loss

tensor([0.7706], grad_fn=<SubBackward0>)

In [59]:
loss.backward() # loss를 미분하여 x와 y의 기울기를 계산
# 편미분: 하나의 변수만 미분하고 나머지 변수는 상수로 취급
print(x.grad, y.grad)  # x 편미분 -1, y 편미분 1

# 손실(예측과 정답의 차이)을 L이라고 가정
# L을 w로 편미분 → w를 어느 쪽으로 고쳐야 손실이 줄어드는지 알음
# L을 b로 편미분 → b도 같은 방식으로 고침
# 그 방향으로 w, b를 조금씩 갱신 (경사 하강)

tensor([-1.]) tensor([1.])


## 8. 다출력 행렬 · `mse_loss`

입력 4 · 출력 3이면 `w`는 `(4, 3)`, `b`는 `(3,)` 입니다.

In [60]:
# y0 = (x0*w00) + (x1*w10) + (x2*w20) + (x3*w30) + b0
# y1 = (x0*w01) + (x1*w11) + (x2*w21) + (x3*w31) + b1
# y2 = (x0*w02) + (x1*w12) + (x2*w22) + (x3*w32) + b2

In [61]:
x = torch.zeros(4) # 4 x 1 행렬 [[0], [0], [0], [0]]
y = torch.ones(3) # 3 x 1 행렬 [[1], [1], [1]]
w = torch.rand(4, 3, requires_grad=True)
# 4 x 3 행렬 [
# [random_w00, random_w01, random_w02],
# [random_w10, random_w11, random_w12],
# [random_w20, random_w21, random_w22],
# [random_w30, random_w31, random_w32]]
b = torch.rand(3, requires_grad=True)
# 3 x 1 행렬 [[random_b0], [random_b1], [random_b2]]

# matmul: 행렬 곱셈
y2 = torch.matmul(x, w) + b
# x의 tensor 는 0이므로 w 행렬 곱셈 결과로 각 tensor 는 0.
# tensor 0 값을 가진 3 x 1 행렬 + b

# y2 = [[
# (0*random_w00) + (0*random_w10) + (0*random_w20) + (0*random_w30) + random_b0,
# (0*random_w01) + (0*random_w11) + (0*random_w21) + (0*random_w31) + random_b1,
# (0*random_w02) + (0*random_w12) + (0*random_w22) + (0*random_w32) + random_b2
# ]]
print(b)
print(y2)

tensor([0.9371, 0.6556, 0.3138], requires_grad=True)
tensor([0.9371, 0.6556, 0.3138], grad_fn=<AddBackward0>)


In [ ]:
# mse_loss: 평균 제곱 오차 손실 함수
loss = F.mse_loss(y2, y)
loss.backward() # loss를 미분하여 w와 b의 기울기를 계산
print(w.grad, b.grad, loss) # w.grad: 4 x 3 행렬, b.grad: 3 x 1 행렬, loss: 평균 제곱 오차 손실
# tensor([
#  [-0., -0., -0.],
#  [-0., -0., -0.],
#  [-0., -0., -0.],
#  [-0., -0., -0.]
# ])
# tensor([-0.0419, -0.2296, -0.4575])
# tensor(0.1978, grad_fn=<MseLossBackward0>)

tensor([[-0., -0., -0.],
        [-0., -0., -0.],
        [-0., -0., -0.],
        [-0., -0., -0.]]) tensor([-0.0419, -0.2296, -0.4575]) tensor(0.1978, grad_fn=<MseLossBackward0>)


---

이론 복습: [`../session08.md`](../session08.md) · 다출력 학습: [`09_multi_output_linear.ipynb`](./09_multi_output_linear.ipynb)